# Modèle TROIS : français + kirundi (Kaggle)

On **continue la lecture du modèle deux** (même tokenizer). Nouveau (`dataset_trois`) : le livre **Dusome 4**.

| # | Changement par rapport au modèle deux |
|---|---|
| 1 | Réponses **sans pénalité de répétition** (1.2 cassait les conjugaisons) |
| 2 | Bible corrigée : 16 000 passages, **moitié vers le français**, longueur comme les tests |
| 3 | Lecture **~6 h** (4 500 pas), Wikipédia FR 001 à **004**, kirundi 1 fois par époque, meilleur point gardé |
| 4 | Discussion **5 époques** (~2,5 h), meilleur point gardé |
| 5 | petite_conversation x30, salutations x50, FLORES dev x5 |
| 6 | **Mêmes tests** que le un et le deux, comparés colonne par colonne |

| Étape | Ce qui se passe |
|---|---|
| 0 | Préparer : trouver les dossiers, l'heure de fin de session |
| 1 | Données publiques (Wikipédia FR 001 à 004, ~20 min) |
| 2 | **Lecture** (~6 h) |
| 3 | **Discussion** (~2,5 h) |
| 4 | Tests : modèle deux et modèle trois, puis comparaison |
| 5 | Essayer le modèle, puis le `.zip` |

**Sauvegardes** : petite sauvegarde tous les 50 pas, copie complète toutes les 4 h,
et **pause propre avant la fin de la session**. Si ce n'est pas fini, la **version suivante** reprend au même pas.

**Avant de lancer (Inputs)** : **Add Input** 2 fois :
1. Datasets > **modele-train** (dernière version). À sa racine : `dataset/`, `dataset_un/`, `dataset_deux/` et `dataset_trois/`.
2. Notebooks > **deux_kaggle** (sa sortie contient `modele_deux/lecture/finale`, le point de départ, et `chat/finale` pour comparer).

**Settings** : Accelerator = **GPU T4 x2**, Internet = **On**. Puis **Save Version > Save & Run All**.

**Session de 12 h** : le notebook s'arrête de lui-même vers 11 h (pause propre), les fichiers restent
dans l'onglet **Output**. Prévu : tout tient dans **une** session (~10 h).
Si ça ne tient pas : Add Input > Notebooks > **ce notebook (trois_kaggle)**, puis Save & Run All : il reprend.

## Étape 0 : préparer Kaggle

In [ ]:
import glob, os, shutil, time
DEBUT = time.time()
HEURES_SESSION = 11.25     # Kaggle coupe à 12 h : on garde du temps pour les tests et la sortie
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Ce que Kaggle a mis dans les Inputs (suit aussi les liens, Kaggle en utilise)
TOUS = []
for racine, dossiers, fichiers in os.walk("/kaggle/input", followlinks=True):
    niveau = racine[len("/kaggle/input"):].count("/")
    if niveau <= 5:
        print("  " * niveau + os.path.basename(racine) + "/", f"({len(fichiers)} fichiers)" if fichiers else "")
    TOUS += [os.path.join(racine, f) for f in fichiers]
print(len(TOUS), "fichiers en tout dans /kaggle/input")

def trouver(fin, quoi, sauf=None):
    # cherche un fichier dont le chemin se termine par `fin` (peu importe la profondeur)
    t = sorted(x for x in TOUS if x.endswith("/" + fin) and not (sauf and sauf in x))
    if not t:
        raise FileNotFoundError(f"\n\nIntrouvable : {quoi} (fichier .../{fin}).\n"
                                f"Clique 'Add Input' et ajoute-le, puis relance. Liste des Inputs ci-dessus.")
    return t[0]

DATASET = os.path.dirname(os.path.dirname(trouver("dialogues/salutations_rn.jsonl", "ancien dataset")))
DATASET_UN = os.path.dirname(os.path.dirname(trouver("insiguro/insiguro.txt", "dataset_un")))
DATASET_DEUX = os.path.dirname(os.path.dirname(trouver("ikibonezamvugo/indimburo_yikirundi.txt", "dataset_deux")))
DATASET_TROIS = os.path.dirname(os.path.dirname(trouver("lecture_rn/dusome4.txt", "dataset_trois")))
# Le modèle deux : sortie du notebook deux_kaggle (modele_deux/lecture/finale)
DEUX = os.path.dirname(os.path.dirname(os.path.dirname(trouver(
    "modele_deux/lecture/finale/config.json",
    "le modèle deux (Add Input > Your Work > Notebooks > deux_kaggle)", sauf="modele_trois"))))
TRAVAIL = "/kaggle/working/modele_trois"
CODE = "/kaggle/working/code_trois"
shutil.copytree(f"{DATASET_TROIS}/code", CODE, dirs_exist_ok=True)

# Reprise : sortie d'une version précédente de CE notebook ajoutée en Input
if not os.path.exists(TRAVAIL):
    anciens = sorted(glob.glob("/kaggle/input/**/modele_trois/lecture", recursive=True))
    if anciens:
        print("Reprise depuis", os.path.dirname(anciens[0]), "(copie, quelques minutes)")
        shutil.copytree(os.path.dirname(anciens[0]), TRAVAIL)
# Wikipédia 001 à 004 : déjà dans un Input s'il existe, sinon téléchargés dans TRAVAIL
_pub = sorted(glob.glob("/kaggle/input/**/publics/lecture_fr/wikipedia_fr_004.txt", recursive=True))
PUBLICS = os.path.dirname(os.path.dirname(_pub[0])) if _pub else f"{TRAVAIL}/publics"

In [ ]:
# 0.b  Commun : variables lues par train.py, heure de fin, état de chaque étape
import torch
os.makedirs(f"{TRAVAIL}/journaux", exist_ok=True)
J = f"{TRAVAIL}/journaux"
os.environ.update(DATASET=DATASET, DATASET_UN=DATASET_UN, DATASET_DEUX=DATASET_DEUX, DATASET_TROIS=DATASET_TROIS,
                  DEUX=DEUX, TRAVAIL=TRAVAIL,
                  PUBLICS=PUBLICS)
if HEURES_SESSION:
    os.environ["FIN_SESSION"] = str(DEBUT + HEURES_SESSION * 3600)
    print("Pause prévue vers", time.strftime("%H:%M", time.localtime(DEBUT + HEURES_SESSION * 3600 - 25 * 60)))
NB_GPU = max(1, torch.cuda.device_count())
LANCER = f"torchrun --nproc_per_node {NB_GPU} train.py" if NB_GPU > 1 else "python -u train.py"
print(f"{NB_GPU} GPU -> {LANCER}")
os.chdir(CODE)
PAUSE = False

def fait(chemin):
    return os.path.exists(chemin)

def apres(etape, dossier):
    """Après une étape : finie, en pause (reprise à la version suivante), ou en échec."""
    global PAUSE
    if fait(f"{dossier}/finale/config.json"):
        print(f"OK : {etape} terminée -> {dossier}/finale")
    elif fait(f"{dossier}/PAUSE"):
        PAUSE = True
        print(f"PAUSE : {etape} pas finie (temps de session). Les étapes suivantes attendent la version suivante.")
    else:
        raise SystemExit(f"ÉCHEC : {etape}. Regarde {J}/{etape}.txt")

def meilleur_modele():
    for ph in ("chat", "lecture"):
        for sous in ("finale", "meilleure"):
            if fait(f"{TRAVAIL}/{ph}/{sous}/config.json"):
                return f"{TRAVAIL}/{ph}/{sous}"
    return f"{DEUX}/lecture/finale"

def modele_deux():
    for sous in ("chat/finale", "chat/meilleure", "lecture/finale"):
        if fait(f"{DEUX}/{sous}/config.json"):
            return f"{DEUX}/{sous}"

print("DATASET    =", DATASET); print("DATASET_UN =", DATASET_UN); print("DATASET_DEUX =", DATASET_DEUX)
print("DATASET_TROIS =", DATASET_TROIS); print("DEUX       =", DEUX)
print("PUBLICS    =", PUBLICS); print("TRAVAIL    =", TRAVAIL)
for nom, ph in [("Lecture", "lecture"), ("Discussion", "chat")]:
    d = f"{TRAVAIL}/{ph}"
    print(f"{nom:11s}: " + ("FINI" if fait(f"{d}/finale") else "en cours (reprise)" if fait(f"{d}/derniere")
                             else "pas commencé"))

## Étape 1 : données publiques
**Quatre** fichiers Wikipédia (001 à 004, 20 M de mots chacun), pour moins répéter les mêmes textes.

In [ ]:
os.chdir(CODE)
if not glob.glob(f"{PUBLICS}/lecture_fr/wikipedia_fr_004.txt"):
    !python -u collecte/telecharger_publics.py --sortie {PUBLICS} --mots 80000000 2>&1 | grep -v "examples/s" | tee {J}/publics.txt
!ls {PUBLICS}/lecture_fr | head -3; ls {PUBLICS}/dialogues

## Étape 2 : lecture (continue la lecture du modèle deux)
4 500 pas (~6 h sur 2 x T4), le meilleur point est gardé. Si la session finit avant, pause propre et reprise.

In [ ]:
os.chdir(CODE)
if not PAUSE and not fait(f"{TRAVAIL}/lecture/finale"):
    !{LANCER} --config configs/trois_lecture.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a {J}/lecture.txt
apres("lecture", f"{TRAVAIL}/lecture")

## Étape 3 : discussion (5 époques, ~2,5 h, le meilleur point est gardé)

In [ ]:
os.chdir(CODE)
if not PAUSE and not fait(f"{TRAVAIL}/chat/finale"):
    !{LANCER} --config configs/trois_chat.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a {J}/chat.txt
if not PAUSE:
    apres("chat", f"{TRAVAIL}/chat")

## Étape 4 : tests
**Exactement les mêmes tests** que le un et le deux (`dialogues_test.jsonl` + FLORES devtest, jamais vus).
Une seule passe, **sans pénalité** (le run deux a montré que c'est mieux). Puis comparaison deux / trois.
`+` = mieux, `-` = moins bien, `=` = pareil (écart de chrF).

In [ ]:
os.chdir(CODE)
if PAUSE:
    print("Tests sautés : l'entraînement n'est pas fini (version suivante).")
else:
    tests = "tests/dialogues_test.jsonl tests/flores.jsonl"
    for nom, modele in [("deux", modele_deux()), ("trois", meilleur_modele())]:
        print("=====", nom, modele)
        !python -u evaluer.py --modele {modele} --tests {tests} --max 100 --montrer 3 \
            --sortie {J}/resultats_{nom}.jsonl 2>&1 | tee {J}/notes_{nom}.txt
    print("\n##### COMPARAISON deux / trois (sans pénalité)")
    !python comparer_resultats.py {J}/resultats_deux.jsonl {J}/resultats_trois.jsonl --noms deux trois | tee {J}/comparaison.txt

## Étape 5 : essayer le modèle

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
appareil = "cuda" if torch.cuda.is_available() else "cpu"
dossier = meilleur_modele()
tok = AutoTokenizer.from_pretrained(dossier)
model = AutoModelForCausalLM.from_pretrained(dossier).to(appareil).eval()
historique = ""

def demander(question, garder=True):
    """Pose une question. garder=True : le modèle se souvient des tours d'avant."""
    global historique
    prompt = f"{historique}<|user|> {question} <|assistant|>"
    ids = tok(prompt, return_tensors="pt", return_token_type_ids=False).to(appareil)
    out = model.generate(**ids, max_new_tokens=150, do_sample=True, temperature=0.3, top_p=0.9,
                         repetition_penalty=1.05, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    reponse = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if garder:
        historique = f"{prompt} {reponse}<|endoftext|>"
    return reponse

print(dossier)
for q in ["Bonjour", "Bwakeye.", "N'amaki?", "Où se trouve le Burundi ?", "Quels sont les pays voisins du Burundi ?",
          "Traduis en français : Umwana n'umukobwa.", "Que veut dire « Guhodoka » en kirundi ?",
          "Ingingo ya 5 y'Ibwirizwa shingiro rya 2018 ivuga iki?", "Que signifie « karangura » en grammaire kirundi ?",
          "Mbwira ingingo ya 26 y'Ibwirizwa shingiro rya 2018 uko yanditse."]:
    print(q, "->", demander(q, garder=False))

## Étape 6 : le `.zip` pour le backend
À mettre dans `litgpt/models/local/`, puis `python scripts/convert_local_model.py <dossier>`.

In [ ]:
if PAUSE:
    print("PAS FINI. Pour continuer : Add Input > Notebooks > trois_kaggle (cette version), puis Save & Run All.")
else:
    dossier_modele = meilleur_modele()
    !cd "{dossier_modele}" && zip -qr /kaggle/working/modele_trois_fr_rn.zip . -x "etat_entrainement.pt"
    !ls -lh /kaggle/working/modele_trois_fr_rn.zip
    print("Le .zip est dans l'onglet Output de cette version.")
print(f"Durée de cette session : {(time.time() - DEBUT) / 3600:.1f} h")